In [ ]:
# 형태소 분석기 Mecab 설치
!pip install konlpy
!pip install mecab-python #표준어에 비표준어모두 추가한것
!bash <(curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh)

In [ ]:
import pandas as pd
from konlpy.tag import Mecab
from gensim.models.doc2vec import TaggedDocument
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 연극

In [ ]:
df = pd.read_excel("/content/drive/MyDrive/캡스톤 디자인/인터파크/연극_TOP70.xlsx")
df=df.drop(['Unnamed: 0'],axis=1)
df=df[['Title', '공연소개', '줄거리', '장르']]
df

,Title,공연소개,줄거리,장르
0,!로맨틱코미디 연극〈운빨로맨스〉! - 대구,NaN,점에 살고 점에 죽는 점보늬의 호랑이띠 남자와 하룻밤 보내기 프로젝트 목숨이 걸린 ...,"로맨스, 코믹, 공감/힐링"
1,(리얼타임 코믹연극) 택시안에서 - 부산,NaN,쾌활하고 유쾌한 택시 운전사 민수 그리고 하영과 소희 두 남녀의 운명적인 만남!\n...,"코믹, 로맨스, 공감/힐링"
2,(리얼타임 코믹연극) 택시안에서 - 서울,NaN,쾌활하고 유쾌한 택시 운전사 민수 그리고 하영과 소희 두 남녀의 운명적인 만남!\n...,"코믹, 로맨스, 공감/힐링"
3,(코믹연극) 달동네-부산,NaN,NaN,"코믹, 감동, 드라마"
4,10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀,"서울 종로구 창신동, 보증금 500-월세 30!\n서울 시내가 다 보이는 옥탑방!\...","이중계약으로 엮인 사랑스러운 두 청춘의, 동화 같은 사랑이 시작된다! 상큼발랄 좌충...","로맨스, 공감/힐링, 판타지"
...,...,...,...,...
65,［서울 대학로］연극 라면,NaN,수학공식처럼 복잡한 사랑에 대해 유쾌하고 코믹하게 풀어보는 90분의 시간\n때는 9...,"로맨스, 코미디, 감동"
66,［연극 on stage］ 99% - 화성,NaN,2008년 세계적인 경제공황의 여파로 위기에 봉착한 한 중견 광고회사는 조직 리모델...,"스릴러, 드라마, 미스터리"
67,［연극 on stage］ 띨뿌리 - 화성,NaN,2008년 세계적인 경제공황의 여파로 위기에 봉착한 한 중견 광고회사는 조직 리모델...,"역사극, 미스터리, 사회극"
68,［연애하기 좋은 날 : 당근거래］ - 전주,NaN,"사귄지 1년 된 지후와 시연은 ""솔로지옥"" 에서 즐겁게 데이트하다 서로에게 상처가 ...","로맨스, 코미디"


In [ ]:
mecab = Mecab()

tagged_corpus_list = []

for index, row in tqdm(df.iterrows(), total=len(df)):
  text = str(row['줄거리'])  # 문자열로 변환
  tag = str(row['Title'])    # 문자열로 변환
  tagged_corpus_list.append(TaggedDocument(tags=[tag], words=mecab.morphs(text)))

print('문서의 수 :', len(tagged_corpus_list))

100%|██████████| 70/70 [00:00<00:00, 790.30it/s]

문서의 수 : 70


In [ ]:
tagged_corpus_list[0]

TaggedDocument(words=['점', '에', '살', '고', '점', '에', '죽', '는', '점', '보늬', '의', '호랑이띠', '남자', '와', '하룻밤', '보내', '기', '프로젝트', '목숨', '이', '걸린', '중요', '한', '미션', '과연', '보늬', '의', '운명', '은', '.', '.?', "'", '일', '주일', '안', '에', '호랑', '이', '며', '남자', '와', '하룻밤', '을', '보내', '지', '않', '으면', '.', '...', '세상', '의', '모든', '불운', '은', '본인', '으로부터', '시작', '된다고', '믿', '는', '운명론자', '점', '보늬', '점', '보늬', '가', '사', '는', '건물', '의', '새', '건물', '주이자', '86', '년', '생', '숫총각', '호랑이띠', "'", '운명', '은', '스스로', '개척', '하', '는', '것', '!', '제발로', '걸', '어', '들어온', '완벽', '한', '부적', '호랑이띠', '남자', '제', '대리', '님', '.', '첫', '눈', '에', '반', '한다는', '말', '.', '전', '믿', '습니다'], tags=['!로맨틱코미디 연극〈운빨로맨스〉! - 대구'])

In [ ]:
from gensim.models import doc2vec

In [ ]:
model = doc2vec.Doc2Vec(vector_size=300, alpha=0.025, min_alpha=0.025, workers=8, window=8)

# Vocabulary 빌드
model.build_vocab(tagged_corpus_list)
# print(f"Tag Size: {len(model.docvecs.doctags.keys())}", end=' / ')

# Doc2Vec 학습
model.train(tagged_corpus_list, total_examples=model.corpus_count, epochs=100)  # 원래 epochs 값 50

# 모델 저장
model.save('dart.doc2vec')

In [ ]:
similar_doc = model.docvecs.most_similar('［서울 대학로］연극 라면')
print(similar_doc)

[('연극 핫식스', 0.573896586894989), ('［2023_셀러브리티 명품 로코］연극 이프온리 크리스마스특별 공연 - 인천', 0.5635914206504822), ('［2023_셀러브리티 명품 로코］연극 이프온리 크리스마스특별 공연 - 대전', 0.5333498120307922), ('［2023_셀러브리티 명품 로코］연극 이프온리 크리스마스특별 공연 - 대학로', 0.524097204208374), ('［2023_셀러브리티 명품 로코］ 연극 이프온리 크리스마스특별 공연 _ 전주', 0.5232043266296387), ('［2023_셀러브리티 명품 로코］연극 이프온리 크리스마스특별 공연 - 대구', 0.5025125741958618), ('［2023_셀러브리티 명품 로코］ 연극 이프온리 크리스마스특별 공연 - 울산', 0.4934164881706238), ('［2023_셀러브리티 명품 로코］연극 이프온리 크리스마스특별 공연 - 광주', 0.48514774441719055), ('［2023_셀러브리티 명품 로코］연극 이프온리 크리스마스특별 공연 - 고양', 0.4784117639064789), ('［2023’셀러브리티 로코연극］러브액츄얼리 첫번째사연_크리스마스 특별공연 - 부산', 0.4427998960018158)]


<ipython-input-49-1c39c4d244fe>:1: DeprecationWarning: Call to deprecated `docvecs` (The `docvecs` property has been renamed `dv`.).
  similar_doc = model.docvecs.most_similar('［서울 대학로］연극 라면')


In [ ]:
similar_doc = model.docvecs.most_similar('(리얼타임 코믹연극) 택시안에서 - 서울')
similar_doc

<ipython-input-50-b62c71955db6>:1: DeprecationWarning: Call to deprecated `docvecs` (The `docvecs` property has been renamed `dv`.).
  similar_doc = model.docvecs.most_similar('(리얼타임 코믹연극) 택시안에서 - 서울')


[('(리얼타임 코믹연극) 택시안에서 - 부산', 0.9887433648109436),
 ('10년 연속 1위 연극〈옥탑방고양이〉- 틴틴홀', 0.528874933719635),
 ('［2023_셀러브리티 명품 로코］연극 이프온리 크리스마스특별 공연 - 인천', 0.5079826712608337),
 ('［2023_셀러브리티 명품 로코］연극 이프온리 크리스마스특별 공연 - 광주', 0.4890398383140564),
 ('［2023_셀러브리티 명품 로코］ 연극 이프온리 크리스마스특별 공연 _ 전주', 0.48274579644203186),
 ('［2023_셀러브리티 명품 로코］연극 이프온리 크리스마스특별 공연 - 대학로', 0.47030752897262573),
 ('［2023_셀러브리티 명품 로코］연극 이프온리 크리스마스특별 공연 - 대구', 0.4687650203704834),
 ('［2023_셀러브리티 명품 로코］연극 이프온리 크리스마스특별 공연 - 대전', 0.4646397829055786),
 ('［2023_셀러브리티 명품 로코］ 연극 이프온리 크리스마스특별 공연 - 울산', 0.4642677307128906),
 ('연극 〈쉬어 매드니스(Shear Madness)〉 - 군포', 0.46082988381385803)]

# 뮤지컬

In [ ]:
df = pd.read_excel("/content/drive/MyDrive/캡스톤 디자인/인터파크/뮤지컬_TOP70.xlsx")
df=df.drop(['Unnamed: 0'],axis=1)
df=df[['Title', '공연소개', '줄거리', '장르']]
df

,Title,공연소개,줄거리,장르
0,"2023 〈스웨그에이지 : 외쳐, 조선〉 - 인천",NaN,시조'가 국가이념인 상상 속의 '조선'\n삶의 고단함과 역경을 시조 속에 담아 훌훌...,"역사극, 판타지"
1,2023 이은결 더 일루션 마스터피스 - 이천,NaN,NaN,"마술, 판타지"
2,2023 창작뮤지컬어워드 NEXT,None,NaN,드라마
3,2023 최현우 Answer - 대구,NaN,"마술사로 걸어온 27년\n2,700여회 공연, 200만 명 관객 돌파\n세계가 인정...","마술, 판타지"
4,2023 최현우 Answer - 성남,NaN,"마술사로 걸어온 27년\n2,700여회 공연, 200만 명 관객 돌파\n세계가 인정...","마술, 판타지"
...,...,...,...,...
64,창작가무극 〈순신〉,최고의 창작진이 선보이는 서울예술단 창작 신작!\n국공립단체로서 한국적 콘텐츠의 가...,"1592년 임진년 4월 어느 날, 순신은 불길한 꿈을 꾼다.\n그의 꿈처럼 임진왜란...","역사극, 드라마"
65,태양의서커스 〈루치아〉,NaN,노란마리골드 꽃밭에 추락한 낙하산\n호기심에 거대한 금속 열쇠를 돌리자 펼쳐지는 꿈...,"판타지, 액션"
66,［뮤지컬 on stage］ 뮤지컬 〈쇼맨 : 어느 독재자의 네 번째 대역배우〉 - 화성,NaN,NaN,"드라마, 공감/힐링"
67,［제주］뉴미디어뮤지컬(노형수퍼마켙),NaN,"시간의 여신, 오늘\n노형수퍼마켙만의 새로운 상설 공연으로 지루할 틈 없이 다양한 ...","판타지, 액션"


In [ ]:
mecab = Mecab()

tagged_corpus_list = []

for index, row in tqdm(df.iterrows(), total=len(df)):
  text = str(row['줄거리'])  # 문자열로 변환
  tag = str(row['Title'])    # 문자열로 변환
  tagged_corpus_list.append(TaggedDocument(tags=[tag], words=mecab.morphs(text)))

print('문서의 수 :', len(tagged_corpus_list))

100%|██████████| 69/69 [00:00<00:00, 919.35it/s]

문서의 수 : 69


In [ ]:
tagged_corpus_list[0]

TaggedDocument(words=['시조', "'", '가', '국가', '이념', '인', '상상', '속', '의', "'", '조선', "'", '삶', '의', '고단', '함', '과', '역경', '을', '시조', '속', '에', '담', '아', '훌훌', '털', '어', '버렸', '던', '백성', '들', '은', '역모', '사건', '으로', '시조', '활동', '이', '금지', '되', '면서', '자유', '도', '행복', '도', '잊', '은', '채', '살아간다', '.', '그러', '던', '중', '15', '년', '만', '에', '누구', '나', '참여', '할', '수', '있', '는', '조선', '시조', '자랑', '이', '열리', '게', '되', '고', ',', '탈', '속', '에', '정체', '를', '감추', '고', '양반', '들', '의', '악행', '을', '파헤쳐', '행복', '한', '세상', '을', '만들', '고자', '조직', '된', '비밀', '시조', '단', '골', '빈', '당', '은', '이것', '을', '기회', '삼', '아', '조선', '에', '새로운', '바람', '을', '불러일으키', '고자', '한다', '.', '한편', ',', '왕', '의', '비선', '실세', '이', '자', '시조', '대판', '서인', '홍국', '은', '자신', '에', '대한', '악덕', '한', '소문', '을', '퍼트리', '고', '다닌다는', '이유', '를', '들', '어', '골', '빈', '당', '을', '잡', '으려는', '음모', '를', '꾸미', '는데', '.', '..'], tags=['2023 〈스웨그에이지 : 외쳐, 조선〉 - 인천'])

In [ ]:
from gensim.models import doc2vec

In [ ]:
model = doc2vec.Doc2Vec(vector_size=300, alpha=0.025, min_alpha=0.025, workers=8, window=8)

# Vocabulary 빌드
model.build_vocab(tagged_corpus_list)
# print(f"Tag Size: {len(model.docvecs.doctags.keys())}", end=' / ')

# Doc2Vec 학습
model.train(tagged_corpus_list, total_examples=model.corpus_count, epochs=100)  # 원래 epochs 값 50

# 모델 저장
model.save('dart.doc2vec')

In [ ]:
similar_doc = model.docvecs.most_similar('2023 창작뮤지컬어워드 NEXT')
print(similar_doc)

[('뮤지컬 〈프리다〉', 0.09805166721343994), ('더 맨 얼라이브 〈초이스〉', 0.07290694862604141), ('라면에 파송송', 0.052421413362026215), ('뮤지컬 〈칠칠〉', 0.049869004637002945), ('뮤지컬 〈쇼맨_어느 독재자의 네 번째 대역배우〉', 0.044780269265174866), ('뮤지컬 〈셜록홈즈 : 앤더슨가의 비밀〉', 0.038362015038728714), ('뮤지컬 〈마리 퀴리〉', 0.03693102300167084), ('뮤지컬 〈레베카〉 10주년 기념공연', 0.029939131811261177), ('태양의서커스 〈루치아〉', 0.028409915044903755), ('뮤지컬 〈인사이드 윌리엄〉 2023', 0.027598153799772263)]


<ipython-input-57-81f28dd506bb>:1: DeprecationWarning: Call to deprecated `docvecs` (The `docvecs` property has been renamed `dv`.).
  similar_doc = model.docvecs.most_similar('2023 창작뮤지컬어워드 NEXT')


In [ ]:
similar_doc = model.docvecs.most_similar('태양의서커스 〈루치아〉')
similar_doc

<ipython-input-58-8a3192fbe5ad>:1: DeprecationWarning: Call to deprecated `docvecs` (The `docvecs` property has been renamed `dv`.).
  similar_doc = model.docvecs.most_similar('태양의서커스 〈루치아〉')


[('2023~24 최현우 Answer', 0.5501459240913391),
 ('2023 최현우 Answer - 대구', 0.5285829305648804),
 ('2023 최현우 Answer - 성남', 0.5203010439872742),
 ('더 맨 얼라이브 〈초이스〉', 0.5146141648292542),
 ('2024 최현우 Answer - 청주', 0.5020354986190796),
 ('2023 최현우 Answer - 안산', 0.49845609068870544),
 ('뮤지컬 〈판〉', 0.494909405708313),
 ('［제주］뉴미디어뮤지컬(노형수퍼마켙)', 0.48967891931533813),
 ('2023 최현우 Answer - 울산', 0.4895908832550049),
 ('뮤지컬 더데빌 : 에덴 (Musical THE DEVIL : EDEN)', 0.47654032707214355)]

In [ ]:
similar_doc = model.docvecs.most_similar('뮤지컬 더데빌 : 에덴 (Musical THE DEVIL : EDEN)')
similar_doc

<ipython-input-59-a9fb0a74be8d>:1: DeprecationWarning: Call to deprecated `docvecs` (The `docvecs` property has been renamed `dv`.).
  similar_doc = model.docvecs.most_similar('뮤지컬 더데빌 : 에덴 (Musical THE DEVIL : EDEN)')


[('2023 매직쇼〈최현우의 미스틱커스〉 - 평택', 0.5375083684921265),
 ('태양의서커스 〈루치아〉', 0.47654035687446594),
 ('뮤지컬 〈타오르는 어둠 속에서〉', 0.4474392235279083),
 ('［제주］뉴미디어뮤지컬(노형수퍼마켙)', 0.43950986862182617),
 ('뮤지컬 〈킴즈〉', 0.4189654290676117),
 ('뮤지컬 〈곤 투모로우〉', 0.4147915542125702),
 ('2023~24 최현우 Answer', 0.3821987807750702),
 ('2023 최현우 Answer - 대구', 0.37528473138809204),
 ('2023 최현우 Answer - 안산', 0.3673877716064453),
 ('2023 최현우 Answer - 울산', 0.3611428439617157)]